# Практическая работа #3: Реляционные связи и объединение данных в Pandas
## Методический ориентир и эталонное решение (для преподавателя)

Данный блокнот содержит:
1. **Описание и реляционную схему базы данных** предметной области (компьютерный клуб).
2. **Методический базис и педагогические акценты**: как наглядно объяснить студентам реляционную модель, первичные/внешние ключи (PK/FK), 4 типа объединения таблиц (`INNER`, `LEFT`, `RIGHT`, `FULL OUTER`), предотвращение декартова произведения и нормализацию данных из JSON.
3. **Сквозное эталонное решение задач всех трех уровней сложности с подробным аналитическим комментарием**:
   - **🟢 Easy**: слияние 3 готовых таблиц через все 4 типа JOIN, анализ лояльности, учет анонимных визитов, выявление простаивающих ПК, аналитика выручки по игровым зонам клуба.
   - **🟡 Medium**: парсинг словарей JSON (`json_normalize`), декомпозиция плоского файла на сущности, аудит ссылочной целостности («битые» ключи), анализ активности клиентов и предпочтений по железу.
   - **🔴 Hard**: распаковка вложенных JSON-массивов списков через `.explode()`, моделирование связи 1:N, построение аналитической схемы «Звезда» из 4 таблиц, маржинальный анализ меню бара, топ-хиты продаж и гипотеза о влиянии бара на игровое время.
4. **Разбор типичных ошибок студентов и критерии оценивания**.


---
# 🗄️ Описание и реляционная схема базы данных

В основе практической работы лежит информационная система киберспортивного компьютерного клуба.
Данные описывают три ключевых бизнес-процесса: **регистрацию и профилирование гостей**, **аренду игровых компьютеров** и **продажи в баре**.

### 1. Архитектурная схема сущностей (Схема «Звезда»)

Связи между сущностями организованы по принципу реляционной модели «Звезда» (Star Schema), где таблицы фактов ссылаются на таблицы измерений:

```text
+---------------------------------------------------------------------------------------------------------+
|                                  РЕЛЯЦИОННАЯ СХЕМА «ЗВЕЗДА» (STAR SCHEMA)                               |
+---------------------------------------------------------------------------------------------------------+

    [dim_users] (Клиенты)                                    [dim_pcs] (Компьютеры)
    +---------------------------------+                      +---------------------------------+
    | PK  user_id      (TEXT)         |                      | PK  pc_id        (TEXT)         |
    +---------------------------------+                      +---------------------------------+
    |     birth_date   (DATE)         |                      |     pc_zone      (TEXT)         |
    |     gender       (TEXT: M/F)    |                      |     hourly_rate  (INT)          |
    |     loyalty_tier (TEXT)         |                      |     gpu          (TEXT)         |
    +----------------+----------------+                      |     cpu          (TEXT)         |
                     |                                       |     ram_gb       (INT)          |
                     | 1:N (user_id)                         |     monitor_hz   (INT)          |
                     |                                       +----------------+----------------+
                     v                                                        |
    +-------------------------------------------------------------------------+ 1:N (pc_id)
    |
    v
    [fact_sessions] (Журнал игровых сессий)
    +---------------------------------+
    | PK  session_id   (INT)          |
    +---------------------------------+
    | FK  user_id      (TEXT)         | ----> dim_users.user_id
    | FK  pc_id        (TEXT)         | ----> dim_pcs.pc_id
    |     date         (DATETIME)     |
    |     hours_played (INT)          |
    |     pc_paid      (INT)          |
    |     discount_promo (INT: 1/0)   |
    +----------------+----------------+
                     |
                     | 1:N (session_id)
                     v
    [fact_bar_sales] (Транзакции бара)                       [dim_catalog] (Каталог бара)
    +---------------------------------+                      +---------------------------------+
    | PK  order_item_id (TEXT)        |                      | PK  item_id      (TEXT)         |
    +---------------------------------+                      +---------------------------------+
    | FK  session_id    (INT)         |                      |     item_name    (TEXT)         |
    | FK  item_id       (TEXT)        | <------------------- |     category     (TEXT)         |
    |     qty           (INT)         |      1:N (item_id)   |     cost_price   (FLOAT)        |
    |     price         (FLOAT)       |                      |     sale_price   (FLOAT)        |
    +---------------------------------+                      +---------------------------------+
```

---

### 2. Спецификация таблиц и полей

| Таблица | Роль сущности | Первичный ключ (PK) | Внешние ключи (FK) | Бизнес-назначение |
|:---|:---|:---:|:---:|:---|
| **`dim_users`** (`users.csv`) | Справочник измерений | `user_id` | — | База зарегистрированных клиентов клуба, уровни карт лояльности, пол и даты рождения. |
| **`dim_pcs`** (`pcs.csv`) | Справочник измерений | `pc_id` | — | Реестр оборудования: зона посадки (Standard, Bootcamp, VIP, PS5), тариф и характеристики комплектующих. |
| **`fact_sessions`** (`sessions.csv`) | Таблица фактов | `session_id` | `user_id`, `pc_id` | Журнал игровых сессий: кто, за каким ПК, когда и сколько времени играл, применялась ли скидка. |
| **`dim_catalog`** (`bar_catalog.csv`) | Справочник измерений | `item_id` | — | Номенклатурный прейскурант бара с закупочной себестоимостью и отпускной ценой для маржинального анализа. |
| **`fact_bar_sales`** (извлекается из JSON) | Таблица фактов | `order_item_id` | `session_id`, `item_id` | Детализированные чеки покупок напитков и снеков, привязанные к конкретным игровым сессиям. |

---

### 3. Особенности структуры данных по уровням сложности

* **🟢 Уровень Easy (3 готовые таблицы):**
  * Студенту даны нормализованные таблицы `sessions.csv`, `users.csv` и `pcs.csv`.
  * Связи: `sessions -> users` (по `user_id`, с наличием анонимных визитов `NaN`), `sessions -> pcs` (по `pc_id`, с наличием простаивающих ПК).
  * Фокус: освоение логики всех 4 типов соединений и агрегации по зонам.

* **🟡 Уровень Medium (2 файла с JSON-словарем):**
  * Характеристики оборудования не вынесены в отдельный файл, а упакованы в JSON-словарь `hardware_specs` внутри строк `sessions_with_specs.csv`.
  * Присутствуют «битые» внешние ключи (в сессиях есть `user_id`, отсутствующие в `users.csv`).
  * Фокус: нормализация (декомпозиция словаря в отдельный справочник `dim_pcs`), очистка журнала от избыточных полей и аудит целостности связей.

* **🔴 Уровень Hard (3 файла с JSON-массивом отношения 1:N):**
  * Покупки в баре упакованы в виде JSON-массива списков `bar_orders` прямо в строках сессий (нарушение 1NF).
  * Фокус: применение метода `.explode()` для разворачивания массива в атомарные строки, генерация синтетического ключа `order_item_id` и построение полноценной схемы «Звезда» с последующим маржинальным анализом через каталог бара.


## 0. Инициализация библиотек и настройка окружения


In [2]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print(f"Версия Pandas: {pd.__version__}")


Версия Pandas: 3.0.1


---
# 📚 Методический раздел: Что объяснить студентам перед началом работы

### 1. Первичные (PK) и внешние (FK) ключи
* **Primary Key (PK)**: уникальный идентификатор строки таблицы (например, `user_id` в таблице пользователей или `pc_id` в таблице ПК). В нем не может быть дубликатов и пропусков.
* **Foreign Key (FK)**: поле в таблице фактов/событий, которое ссылается на Primary Key другой таблицы (например, `user_id` в таблице сессий). На один `user_id` может приходиться много сессий (связь 1:N).

### 2. Типичная ошибка студентов: Декартово произведение (Cartesian Explosion)
* Если в справочнике по ключу объединения есть дубликаты, строки начнут перемножаться: $N \times M$.
* **Правило:** перед объединением студент обязан проверить уникальность ключа в справочной таблице: `assert df_users['user_id'].is_unique`.

### 3. Все 4 типа JOIN:
* **`INNER`**: оставляем только взаимные совпадения (сессии подтвержденных пользователей).
* **`LEFT`**: левая таблица главная. Сохраняем все сессии, даже если клиента нет в CRM (`user_id = NaN`).
* **`RIGHT`**: правая таблица главная. Сохраняем все сущности справочника, даже если в журнале сессий о них нет записей (находим неиспользуемые ПК, клиентов без визитов).
* **`OUTER`**: сквозной аудит с двух сторон. Сохраняем все строки, подставляя `NaN` там, где нет соответствия.


---
# 🟢 Часть 1. Разбор уровня сложности EASY (Базовый)
**Целевые навыки:** работа с 3 готовыми связанными CSV-файлами, определение PK/FK, последовательное применение всех 4 типов соединений (`inner`, `left`, `right`, `outer`), бизнес-интерпретация результатов, сквозной расчет выручки по зонам клуба.


In [3]:
# 1.1. Загрузка трех таблиц варианта 01
easy_dir = 'easy/variant_01'
df_sess_easy = pd.read_csv(f'{easy_dir}/sessions.csv', sep=';')
df_users_easy = pd.read_csv(f'{easy_dir}/users.csv', sep=';')
df_pcs_easy = pd.read_csv(f'{easy_dir}/pcs.csv', sep=';')

print("Размерности таблиц:")
print(f"Сессии: {df_sess_easy.shape}")
print(f"Пользователи: {df_users_easy.shape}")
print(f"Компьютеры: {df_pcs_easy.shape}")

print("\nПроверка уникальности первичных ключей:")
print(f"user_id в users уникален: {df_users_easy['user_id'].is_unique}")
print(f"pc_id в pcs уникален: {df_pcs_easy['pc_id'].is_unique}")


Размерности таблиц:
Сессии: (3015, 6)
Пользователи: (500, 4)
Компьютеры: (40, 4)

Проверка уникальности первичных ключей:
user_id в users уникален: True
pc_id в pcs уникален: True


### Разбор шага 1.1: Проверка уникальности ключей
* **Методический акцент:** Обратите внимание студентов, что `user_id` в таблице `sessions` **не должен** быть уникальным, так как это внешний ключ связи 1:N (один клиент играет много сессий). Уникальность обязательна только в справочниках (`users`, `pcs`).
* **Типичная ошибка:** Студент проверяет `is_unique` на таблице сессий и удивляется результату `False`.


In [4]:
# 1.2. Шаг 2.1: INNER JOIN — Анализ сессий зарегистрированных клиентов
inner_sess_users = pd.merge(df_sess_easy, df_users_easy, on='user_id', how='inner')

print(f"Всего сессий в журнале: {len(df_sess_easy)}")
print(f"Сессий зарегистрированных клиентов (INNER): {len(inner_sess_users)}")

# Анализ времени игры по статусам лояльности
loyalty_stats = inner_sess_users.groupby('loyalty_tier').agg(
    sessions=('session_id', 'count'),
    avg_hours=('hours_played', 'mean'),
    total_hours=('hours_played', 'sum')
).sort_values('avg_hours', ascending=False)

print("\nСтатистика по статусам лояльности клиентов:")
display(loyalty_stats)


Всего сессий в журнале: 3015
Сессий зарегистрированных клиентов (INNER): 2798

Статистика по статусам лояльности клиентов:


,sessions,avg_hours,total_hours
loyalty_tier,,,
Bronze,1393,2.38,3309
Silver,856,2.36,2023
Gold,475,2.29,1090
Platinum,74,2.22,164


### Разбор шага 1.2: INNER JOIN и поведение клиентов
* **Методический акцент:** `INNER JOIN` строго отсекает все записи, где `user_id` не сошелся (анонимные визиты). Это инструмент для анализа поведения именно постоянной клубной аудитории.
* **Аналитический вывод:** Клиенты более высоких уровней лояльности (Platinum, Gold) проводят в клубе существенно больше времени за одну сессию, чем обладатели Bronze.


In [5]:
# 1.3. Шаг 2.2: LEFT JOIN — Учет анонимных визитов
left_sess_users = pd.merge(df_sess_easy, df_users_easy, on='user_id', how='left')

anon_sessions = left_sess_users[left_sess_users['user_id'].isna()]
anon_count = len(anon_sessions)
anon_share = anon_count / len(left_sess_users) * 100

print(f"Количество анонимных визитов (без клубной карты): {anon_count} ({anon_share:.1f}%)")

# Сравнение длительности сессий клиентов и анонимных гостей
left_sess_users['client_type'] = np.where(left_sess_users['user_id'].isna(), 'Анонимный гость', 'Клубный клиент')
client_comparison = left_sess_users.groupby('client_type')['hours_played'].agg(['count', 'mean', 'median'])
print("\nСравнение клиентов клуба и анонимных гостей:")
display(client_comparison)


Количество анонимных визитов (без клубной карты): 217 (7.2%)

Сравнение клиентов клуба и анонимных гостей:


,count,mean,median
client_type,,,
Анонимный гость,217,2.28,2.00
Клубный клиент,2798,2.35,2.00


### Разбор шага 1.3: LEFT JOIN и анонимные визиты
* **Методический акцент:** В бизнес-аналитике факт события нельзя терять. Если бы аналитик считал посещаемость через `inner join`, он потерял бы от 15% до 25% всех визитов.
* **Типичная ошибка:** Студент удаляет строки с `NaN` через `.dropna()`, теряя тем самым анонимную выручку клуба.


In [6]:
# 1.4. Шаг 2.3: RIGHT JOIN — Аудит простаивающего оборудования клуба
right_sess_pcs = pd.merge(df_sess_easy, df_pcs_easy, on='pc_id', how='right')

# Компьютеры, на которых за месяц не было ни одной сессии
idle_pcs = right_sess_pcs[right_sess_pcs['session_id'].isna()][['pc_id', 'pc_zone', 'hourly_rate', 'gpu']].drop_duplicates()
print(f"Обнаружено простаивающих ПК (без сессий за месяц): {len(idle_pcs)}")
display(idle_pcs)


Обнаружено простаивающих ПК (без сессий за месяц): 3


,pc_id,pc_zone,hourly_rate,gpu
3015,PC-38,PS5,250,Sony PlayStation 5 Slim
3016,PC-39,PS5,250,Sony PlayStation 5
3017,PC-40,PS5,250,Sony PlayStation 5 Slim


### Разбор шага 1.4: RIGHT JOIN и поиск неиспользуемых ресурсов
* **Методический акцент:** `RIGHT JOIN` сохраняет все сущности правого справочника (`pcs`). Фильтр `session_id.isna()` мгновенно выявляет машины, на которые не было посадки.
* **Бизнес-интерпретация:** Простаивающие ПК могут быть на техническом обслуживании или в резерве.


In [7]:
# 1.5. Шаг 2.4: FULL OUTER JOIN — Сквозной аудит клиентов и посещений
outer_sess_users = pd.merge(df_sess_easy, df_users_easy, on='user_id', how='outer')

# Клиенты, которые зарегистрированы, но не посещали клуб в этом месяце
inactive_users = outer_sess_users[outer_sess_users['session_id'].isna()]
print(f"Зарегистрированных клиентов без единого визита за месяц: {len(inactive_users)}")
print(f"Доля неактивных клиентов (отток): {len(inactive_users) / len(df_users_easy) * 100:.1f}%")


Зарегистрированных клиентов без единого визита за месяц: 61
Доля неактивных клиентов (отток): 12.2%


### Разбор шага 1.5: FULL OUTER JOIN и клиентский отток
* **Методический акцент:** Полное внешнее соединение сопоставляет обе базы разом. Записи с `session_id is NaN` — это «спящие» клиенты, по которым маркетинг может запускать реактивационные рассылки.


In [8]:
# 1.6. Шаг 3: Сквозной анализ выручки по зонам клуба
df_full_easy = df_sess_easy.merge(df_users_easy, on='user_id', how='inner') \
                           .merge(df_pcs_easy, on='pc_id', how='inner')

# Расчет оплаты аренды с учетом скидочного промокода (10%)
df_full_easy['rental_paid'] = (
    df_full_easy['hours_played'] * df_full_easy['hourly_rate'] * 
    np.where(df_full_easy['discount_promo'] == 1, 0.9, 1.0)
).astype(int)

# 1. Аналитика по игровым зонам клуба
zone_stats = df_full_easy.groupby('pc_zone').agg(
    total_sessions=('session_id', 'count'),
    total_hours=('hours_played', 'sum'),
    total_revenue=('rental_paid', 'sum'),
    avg_check=('rental_paid', 'mean')
).sort_values('total_revenue', ascending=False)

print("Анализ игровых зон клуба (выручка, часы, средний чек):")
display(zone_stats)

# 2. Распределение клиентов разной лояльности по зонам клуба
loyalty_by_zone = pd.crosstab(
    df_full_easy['pc_zone'], 
    df_full_easy['loyalty_tier'], 
    margins=True, 
    margins_name='Всего'
)
print("\nРаспределение сессий по зонам и уровням лояльности:")
display(loyalty_by_zone)


Анализ игровых зон клуба (выручка, часы, средний чек):


,total_sessions,total_hours,total_revenue,avg_check
pc_zone,,,,
Standard,1524,3589,352900,231.56
Bootcamp,738,1752,258765,350.63
VIP,462,1078,210060,454.68
PS5,74,167,41200,556.76



Распределение сессий по зонам и уровням лояльности:


loyalty_tier,Bronze,Gold,Platinum,Silver,Всего
pc_zone,,,,,
Bootcamp,373,125,21,219,738
PS5,35,12,2,25,74
Standard,736,282,38,468,1524
VIP,249,56,13,144,462
Всего,1393,475,74,856,2798


### Разбор шага 1.6: Аналитика зон клуба
* **Методический акцент:** Простые группировки по игровым зонам (`pc_zone`) позволяют студентам увидеть бизнес-структуру: зоны с более высокими тарифами (VIP, Bootcamp) формируют значительную часть выручки даже при меньшем числе мест.


---
# 🟡 Часть 2. Разбор уровня сложности MEDIUM (Средний)
**Целевые навыки:** распаковка вложенных JSON-словарей через `pd.json_normalize()`, декомпозиция плоского файла на сущности (нормализация 3NF), аудит реляционной целостности и «висячих» внешних ключей (`left`/`right`/`outer`), анализ предпочтений активных игроков.


In [9]:
# 2.1. Загрузка данных Medium варианта 01
med_dir = 'medium/variant_01'
df_sess_med = pd.read_csv(f'{med_dir}/sessions_with_specs.csv', sep=';')
df_users_med = pd.read_csv(f'{med_dir}/users.csv', sep=';')

print(f"Размерность сессий: {df_sess_med.shape}")
print(f"Размерность пользователей: {df_users_med.shape}")
print("\nПример строки с JSON hardware_specs:")
print(df_sess_med[['session_id', 'pc_id', 'hardware_specs']].head(1).to_dict(orient='records'))


Размерность сессий: (3520, 10)
Размерность пользователей: (600, 4)

Пример строки с JSON hardware_specs:
[{'session_id': 'SESS-20000', 'pc_id': 'PC-19', 'hardware_specs': '{"gpu": "RTX 3050", "cpu": "i5-12400F", "ram_gb": 16, "monitor_hz": 165}'}]


### Разбор шага 2.1: Денормализация в реальных данных
* **Методический акцент:** В реальных системах события часто логируются в виде плоского JSON. Задача дата-инженера — выделить повторяющиеся сущности в нормализованный справочник.


In [10]:
# 2.2. Декомпозиция: распаковка JSON и формирование отдельной таблицы dim_pcs
specs_parsed = [json.loads(s) for s in df_sess_med['hardware_specs']]
df_specs = pd.json_normalize(specs_parsed)

df_pcs_combined = pd.concat([df_sess_med[['pc_id', 'pc_zone']], df_specs], axis=1)

# Выделение независимой таблицы измерений компьютеров dim_pcs
dim_pcs = df_pcs_combined.drop_duplicates(subset=['pc_id']).reset_index(drop=True)
print(f"Сформирована таблица dim_pcs (уникальных компьютеров: {len(dim_pcs)}):")
display(dim_pcs.head())

# Очистка таблицы сессий: удаляем hardware_specs и pc_zone (атрибут ПК теперь в dim_pcs)
fact_sessions_med = df_sess_med.drop(columns=['hardware_specs', 'pc_zone']).copy()
print(f"\nТаблица fact_sessions избавлена от избыточных полей, размерность: {fact_sessions_med.shape}")


Сформирована таблица dim_pcs (уникальных компьютеров: 46):


,pc_id,pc_zone,gpu,cpu,ram_gb,monitor_hz
0,PC-19,Standard,RTX 3050,i5-12400F,16,165
1,PC-21,Standard,GTX 1660 Super,i5-10400F,16,144
2,PC-14,Standard,GTX 1660 Super,i5-10400F,16,144
3,PC-22,Standard,GTX 1660 Super,i5-10400F,16,144
4,PC-01,Standard,RTX 3050,i5-12400F,16,165



Таблица fact_sessions избавлена от избыточных полей, размерность: (3520, 8)


### Разбор шага 2.2: Нормализация и дедупликация
* **Методический акцент:** Обратите внимание на `drop_duplicates(subset=['pc_id'])`. Если не удалить дубликаты компьютеров из сформированного справочника, последующее слияние приведет к перемножению строк (декартов взрыв).


In [11]:
# 2.3. Аудит ссылочной целостности через 4 типа соединений

# 1) LEFT JOIN: поиск "висячих" ключей (сессии с несуществующими user_id)
left_audit = pd.merge(fact_sessions_med, df_users_med, on='user_id', how='left')
orphaned = left_audit[left_audit['user_id'].notna() & left_audit['gender'].isna()]
anon = left_audit[left_audit['user_id'].isna()]

print(f"Сессий с 'висячими' ключами (user_id нет в базе): {len(orphaned)}")
print(f"Полностью анонимных сессий (user_id is NaN): {len(anon)}")

# 2) INNER JOIN: расчет средней выручки по подтвержденным профилям
inner_clean = pd.merge(fact_sessions_med, df_users_med, on='user_id', how='inner')
inner_clean['total_revenue'] = inner_clean['total_paid'] + inner_clean['extra_bar_spent']
rev_by_loyalty = inner_clean.groupby('loyalty_tier')['total_revenue'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False)
print("\nВыручка по статусам лояльности (INNER JOIN):")
display(rev_by_loyalty)

# 3) RIGHT JOIN: поиск неактивных клиентов (отток)
right_audit = pd.merge(fact_sessions_med, df_users_med, on='user_id', how='right')
churn_users = right_audit[right_audit['session_id'].isna()]
print(f"\nЗарегистрированных клиентов без единой сессии: {len(churn_users)}")

# 4) FULL OUTER JOIN: сводная матрица сверки ссылочной целостности базы
outer_reconciliation = pd.merge(fact_sessions_med, df_users_med, on='user_id', how='outer', indicator=True)
print("\nСводная матрица ссылочной целостности (OUTER JOIN с indicator=True):")
recon_counts = outer_reconciliation['_merge'].value_counts()
recon_df = pd.DataFrame({
    'Тип записи': [
        'Подтвержденные сессии клиентов (both)',
        'Анонимные и битые сессии (left_only)',
        'Клиенты без визитов за месяц (right_only)'
    ],
    'Количество': [
        recon_counts.get('both', 0),
        recon_counts.get('left_only', 0),
        recon_counts.get('right_only', 0)
    ]
})
display(recon_df)


Сессий с 'висячими' ключами (user_id нет в базе): 160
Полностью анонимных сессий (user_id is NaN): 230

Выручка по статусам лояльности (INNER JOIN):


,count,mean,sum
loyalty_tier,,,
Bronze,1031,611.13,630070
Standard,1313,545.09,715705
Silver,488,541.55,264275
VIP,105,526.14,55245
Gold,193,516.94,99770



Зарегистрированных клиентов без единой сессии: 71

Сводная матрица ссылочной целостности (OUTER JOIN с indicator=True):


,Тип записи,Количество
0,Подтвержденные сессии клиентов (both),3130
1,Анонимные и битые сессии (left_only),390
2,Клиенты без визитов за месяц (right_only),71


### Разбор шага 2.3: Аудит целостности и «висячие» ключи
* **Методический акцент:** В Medium добавлен кейс «битых» ссылок: в `sessions` указан `user_id`, которого нет в таблице `users` (например, удаленный аккаунт).
* Параметр `indicator=True` — лучший инструмент для быстрой инспекции чистоты слияния.


In [12]:
# 2.4. Шаг 3: Анализ активности клиентов и зон посадки
merged_med = fact_sessions_med.merge(df_users_med, on='user_id', how='inner') \
                              .merge(dim_pcs, on='pc_id', how='inner')

# 1. Топ-5 самых активных клиентов по сыгранному времени
top5_players = merged_med.groupby(['user_id', 'loyalty_tier']).agg(
    total_hours=('hours_played', 'sum'),
    total_sessions=('session_id', 'count'),
    total_spent=('total_paid', 'sum')
).sort_values('total_hours', ascending=False).head(5)

print("Топ-5 самых активных клиентов клуба:")
display(top5_players)

# 2. Предпочтения топ-5 клиентов по зонам и видеокартам
top5_ids = top5_players.index.get_level_values('user_id')
top5_sessions = merged_med[merged_med['user_id'].isin(top5_ids)]
top5_preferences = pd.crosstab(top5_sessions['user_id'], top5_sessions['pc_zone'])
print("\nВ каких зонах предпочитают играть лидеры клуба:")
display(top5_preferences)

# 3. Средние расходы на бар по зонам клуба
bar_by_zone = merged_med.groupby('pc_zone').agg(
    avg_bar_spent=('extra_bar_spent', 'mean'),
    median_bar_spent=('extra_bar_spent', 'median'),
    total_bar_spent=('extra_bar_spent', 'sum')
).sort_values('avg_bar_spent', ascending=False)
print("\nТраты в баре по зонам посадки:")
display(bar_by_zone)


Топ-5 самых активных клиентов клуба:


,,total_hours,total_sessions,total_spent
user_id,loyalty_tier,,,
USR-2210,Standard,49,12,5620
USR-2474,Bronze,48,15,6075
USR-2147,Standard,48,15,6760
USR-2212,Bronze,44,13,5560
USR-2166,Bronze,43,11,5350



В каких зонах предпочитают играть лидеры клуба:


pc_zone,Bootcamp,Standard,VIP
user_id,,,
USR-2147,3,7,5
USR-2166,7,4,0
USR-2210,1,8,3
USR-2212,4,6,3
USR-2474,6,6,3



Траты в баре по зонам посадки:


,avg_bar_spent,median_bar_spent,total_bar_spent
pc_zone,,,
Bootcamp,228.97,120.00,221640
VIP,201.17,120.00,102800
Standard,193.94,120.00,320200


### Разбор шага 2.4: Анализ потребительского поведения
* **Методический акцент:** Вместо сложной абстрактной фильтрации студенты находят реальных лидеров клуба, их предпочтения по зонам и выявляют зоны с максимальным потреблением бара.


---
# 🔴 Часть 3. Разбор уровня сложности HARD (Продвинутый)
**Целевые навыки:** моделирование связи 1:N, распаковка массивов списков через `.explode()`, сборка реляционной схемы «Звезда» из 4 таблиц, все 4 типа JOIN, маржинальный анализ продаж бара, топ-хиты товаров и сопоставление игрового времени.


In [13]:
# 3.1. Загрузка данных Hard варианта 01
hard_dir = 'hard/variant_01'
df_sess_hard = pd.read_csv(f'{hard_dir}/sessions_with_orders.csv', sep=';')
df_users_hard = pd.read_csv(f'{hard_dir}/users.csv', sep=';')
df_cat_hard = pd.read_csv(f'{hard_dir}/bar_catalog.csv', sep=';')

print(f"Сессии: {df_sess_hard.shape}")
print(f"Пользователи: {df_users_hard.shape}")
print(f"Каталог товаров бара: {df_cat_hard.shape}")

print("\nПример чека бара в строке сессии:")
print(df_sess_hard['bar_orders'].iloc[0])


Сессии: (4525, 9)
Пользователи: (750, 4)
Каталог товаров бара: (20, 5)

Пример чека бара в строке сессии:
[{"item_id": "BAR-14", "qty": 2, "price": 110}, {"item_id": "BAR-06", "qty": 2, "price": 160}]


### Разбор шага 3.1: Нарушение 1NF (Первой нормальной формы)
* **Методический акцент:** Хранение списка или массива в ячейке таблицы нарушает требование атомарности значений (1NF). Такую структуру невозможно напрямую связать с номенклатурным каталогом бара.


In [22]:
# 3.2. Распаковка JSON-массивов 1:N через .explode() и создание fact_bar_sales
df_sess_orders = df_sess_hard[['session_id', 'bar_orders']].copy()

# Парсинг строки JSON в Python-список
df_sess_orders['bar_orders'] = df_sess_orders['bar_orders'].apply(lambda s: json.loads(s) if pd.notna(s) else [])

# Распаковка списка в отдельные строки
df_exploded = df_sess_orders.explode('bar_orders').dropna(subset=['bar_orders'])
# Отфильтровываем сессии с пустым чеком []
df_exploded = df_exploded[df_exploded['bar_orders'].apply(lambda x: isinstance(x, dict))]

# Извлечение полей словаря
orders_data = list(df_exploded['bar_orders'])
df_items = pd.DataFrame(orders_data, index=df_exploded.index)

display(df_items)

# Формирование нормализованной таблицы fact_bar_sales
fact_bar_sales = pd.DataFrame({
    'order_item_id': [f"{sid}-{i}" for i, sid in enumerate(df_exploded['session_id'])],
    'session_id': df_exploded['session_id'].values,
    'item_id': df_items['item_id'].values,
    'qty': df_items['qty'].astype(int).values,
    'price': df_items['price'].astype(float).values
})

print(f"Успешно распаковано строк заказов бара: {len(fact_bar_sales)}")
display(type(fact_bar_sales))

# Таблица fact_sessions (без тяжелого JSON)
fact_sessions_hard = df_sess_hard.drop(columns=['bar_orders']).copy()


,item_id,qty,price
0,BAR-14,2,110
0,BAR-06,2,160
1,BAR-06,2,160
3,BAR-05,1,70
3,BAR-04,1,100
...,...,...,...
4523,BAR-11,1,220
4523,BAR-01,1,120
4524,BAR-02,1,200
4524,BAR-12,2,260


Успешно распаковано строк заказов бара: 4578


pandas.DataFrame

### Разбор шага 3.2: Превращение массива в строки через `.explode()`
* **Методический акцент:** Метод `.explode()` размножает родительскую строку на столько копий, сколько элементов в массиве. Получается реляционная схема **«Звезда»**:
  - `dim_users` (клиенты)
  - `dim_catalog` (номенклатура бара)
  - `fact_sessions` (журнал сессий)
  - `fact_bar_sales` (транзакции бара)


In [15]:
# 3.3. Применение 4 типов соединений в Hard

# 1) LEFT JOIN: честный расчет совокупной выручки (не теряя сессии без покупок в баре)
fact_bar_sales['item_sum'] = fact_bar_sales['qty'] * fact_bar_sales['price']
bar_totals = fact_bar_sales.groupby('session_id')['item_sum'].sum().reset_index(name='bar_total')

sessions_with_rev = pd.merge(fact_sessions_hard, bar_totals, on='session_id', how='left')
sessions_with_rev['bar_total'] = sessions_with_rev['bar_total'].fillna(0)
sessions_with_rev['session_revenue'] = sessions_with_rev['pc_paid'] + sessions_with_rev['bar_total']

total_club_rev = sessions_with_rev['session_revenue'].sum()
bar_rev = sessions_with_rev['bar_total'].sum()
print(f"Общая выручка клуба: {total_club_rev:,.0f} руб.")
print(f"Выручка бара: {bar_rev:,.0f} руб. ({bar_rev / total_club_rev * 100:.1f}%)")

# 2) RIGHT JOIN: аудит неликвидных позиций каталога бара
cat_audit = pd.merge(fact_bar_sales, df_cat_hard, on='item_id', how='right')
zero_sales_items = cat_audit[cat_audit['order_item_id'].isna()][['item_id', 'item_name', 'category', 'sale_price']].drop_duplicates()
print(f"\nПозиций каталога без единой продажи (неликвид): {len(zero_sales_items)}")
display(zero_sales_items)

# 3) INNER JOIN: маржинальный анализ категорий бара
margin_df = pd.merge(fact_bar_sales, df_cat_hard, on='item_id', how='inner')
margin_df['revenue'] = margin_df['qty'] * margin_df['sale_price']
margin_df['profit'] = margin_df['qty'] * (margin_df['sale_price'] - margin_df['cost_price'])

cat_summary = margin_df.groupby('category').agg(
    units_sold=('qty', 'sum'),
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum')
).sort_values('profit', ascending=False)
print("\nМаржинальность категорий бара:")
display(cat_summary)

# 4) FULL OUTER JOIN: аудит пользовательской базы
user_audit = pd.merge(fact_sessions_hard, df_users_hard, on='user_id', how='outer')
print(f"\nАнонимных визитов без аккаунта: {user_audit['user_id'].isna().sum()}")
print(f"Зарегистрированных клиентов без визитов: {user_audit['session_id'].isna().sum()}")


Общая выручка клуба: 2,605,765 руб.
Выручка бара: 906,040 руб. (34.8%)



Позиций каталога без единой продажи (неликвид): 4


,item_id,item_name,category,sale_price
4578,BAR-17,Чай Матча Латте 0.3л,Hot Drinks,250
4579,BAR-18,Протеиновый батончик Bombbar,Sweets,210
4580,BAR-19,Смузи Ягодный 0.33л,Drinks,310
4581,BAR-20,Фруктовые чипсы Яблоко 40г,Snacks,190



Маржинальность категорий бара:


,units_sold,revenue,profit
category,,,
Drinks,2494,380430,176865
Snacks,1407,225080,103110
Food,934,161800,75560
Hot Drinks,500,63760,48340
Sweets,833,74970,37485



Анонимных визитов без аккаунта: 209
Зарегистрированных клиентов без визитов: 72


### Разбор шага 3.3: 4 типа слияния в сложной схеме
* **Логика `LEFT JOIN` + `fillna(0)`:** Критический момент! Если объединить сессии с продажами бара через `inner`, пропадут 30-40% сессий (где не покупали в баре). Итоговая выручка клуба окажется занижена.
* **Логика `RIGHT JOIN`:** Выявляет позиции каталога, которые не продавались.
* **Логика `INNER JOIN`:** Маржинальность рассчитывается по фактически проданным позициям.


In [16]:
# 3.4. Шаг 3: Анализ потребительского поведения и продаж бара

# 1. Топ-3 товара по продажам (штуки) и топ-3 по валовой прибыли
items_stats = pd.merge(fact_bar_sales, df_cat_hard, on='item_id', how='inner')
items_stats['total_profit'] = items_stats['qty'] * (items_stats['sale_price'] - items_stats['cost_price'])

top_units = items_stats.groupby('item_name')['qty'].sum().sort_values(ascending=False).head(3)
top_profit = items_stats.groupby('item_name')['total_profit'].sum().sort_values(ascending=False).head(3)

print("Топ-3 товара бара по количеству проданных штук:")
display(top_units.to_frame('Продано (шт)'))

print("\nТоп-3 товара бара по принесенной валовой прибыли:")
display(top_profit.to_frame('Прибыль (руб)'))

# 2. Средний чек бара по игровым зонам клуба
zone_bar = sessions_with_rev.groupby('pc_zone').agg(
    sessions=('session_id', 'count'),
    bar_revenue=('bar_total', 'sum'),
    avg_bar_check=('bar_total', 'mean')
).sort_values('avg_bar_check', ascending=False)
print("\nПотребление бара по зонам клуба:")
display(zone_bar)

# 3. Проверка гипотезы: связь заказа в баре и длительности игры
sessions_with_rev['has_bar_order'] = np.where(sessions_with_rev['bar_total'] > 0, 'С заказом в баре', 'Без заказа в баре')
duration_comparison = sessions_with_rev.groupby('has_bar_order')['hours_played'].agg(
    sessions_count='count',
    avg_hours='mean',
    median_hours='median'
)
print("\nСравнение длительности игры гостей с покупками в баре и без них:")
display(duration_comparison)


Топ-3 товара бара по количеству проданных штук:


,Продано (шт)
item_name,
Энергетик Flash Up 0.45л,839
Энергетик Monster Energy 0.5л,601
Чипсы Lays Сметана-Зелень 140г,577



Топ-3 товара бара по принесенной валовой прибыли:


,Прибыль (руб)
item_name,
Энергетик Monster Energy 0.5л,54090
Энергетик Red Bull 0.33л,48510
Энергетик Flash Up 0.45л,46145



Потребление бара по зонам клуба:


,sessions,bar_revenue,avg_bar_check
pc_zone,,,
VIP,825,173870.00,210.75
Standard,2245,449410.00,200.18
PS5,314,61570.00,196.08
Bootcamp,1141,221190.00,193.86



Сравнение длительности игры гостей с покупками в баре и без них:


,sessions_count,avg_hours,median_hours
has_bar_order,,,
Без заказа в баре,1804,2.56,2.00
С заказом в баре,2721,2.85,2.00


### Разбор шага 3.4: Понятные и живые инсайты
* **Товары-локомотивы:** Студенты сразу видят разницу между объемом продаж (штуки) и маржинальностью (прибыль) — например, энергетики могут продаваться чаще, но снеки или премиальные напитки приносить больше маржи.
* **Поведение гостей:** Сравнение длительности сессий подтверждает гипотезу: гости, покупающие напитки и закуски, в среднем проводят за игрой больше времени.


---
# 🎓 Рекомендации для преподавателя по проверке работ

### Чек-лист критериев оценки Практической работы №3:
1. **Понимание реляционной модели и первичных ключей:**
   - Проверяет ли студент уникальность ключей перед слиянием (`is_unique`, `.value_counts().max() == 1`).
   - Понимает ли студент, почему недопустимо декартово произведение при объединении.
2. **Осознанное использование 4 типов соединений:**
   - **`INNER`**: обоснованное отсечение пустых связей;
   - **`LEFT`**: сохранение всех сессий (понимание того, что `NaN` в профиле — это не ошибка данных, а признак анонимного гостя);
   - **`RIGHT`**: умение анализировать записи правой таблицы, для которых нет совпадений в левой (простаивающие ПК, клиенты без визитов);
   - **`OUTER`**: проведение полного перекрестного аудита.
3. **Корректность работы с ненормализованными данными (Medium/Hard):**
   - Умение распарсить JSON через `pd.json_normalize` или `json.loads`;
   - Владение методом `.explode()` для разворачивания списков в атомарные строки;
   - Формирование правильной транзакционной таблицы с внешними ключами.
4. **Аналитические выводы:**
   - Ответы на вопросы задания сформулированы словами в Markdown-ячейках, а не только оставлены в виде сырого кода.
